In [10]:
'''
PyTorch 1.0 强化学习 (Packt Publishing)
第2章：马尔可夫决策过程与动态规划
作者：Yuxi (Hayden) Liu
修改说明：升级至 gymnasium 库 (FrozenLake-v1)，适配新版 reset/step 返回值，增加详细中文注释与输出
'''

import gymnasium as gym  # 使用新版 gymnasium 库
import torch

# ==========================================
# 1. 环境初始化与基础交互
# ==========================================

# 创建 FrozenLake-v1 环境，render_mode="ansi" 可以在终端直接打印字符地图
env = gym.make("FrozenLake-v1", render_mode="ansi")

# 获取状态空间和动作空间的维度
n_state = env.observation_space.n
print(f"【环境信息】状态总数: {n_state}")

n_action = env.action_space.n
print(f"【环境信息】动作总数: {n_action} (0:左, 1:下, 2:右, 3:上)")

# 重置环境，新版 API 返回 (初始状态, 信息字典)
state, info = env.reset()
print("\n【初始状态地图】")
print(env.render())

# 执行一步动作（动作 1 代表向下），新版 API 返回 5 个值
new_state, reward, terminated, truncated, info = env.step(1)
print("\n【执行动作 1 (向下) 后的地图】")
print(env.render())

print(f"  新状态索引: {new_state}")
print(f"  获得奖励: {reward}")
print(f"  是否自然终止 (terminated): {terminated}")
print(f"  是否超时截断 (truncated): {truncated}")
print(f"  额外信息 (info): {info}")

【环境信息】状态总数: 16
【环境信息】动作总数: 4 (0:左, 1:下, 2:右, 3:上)

【初始状态地图】

SFFF
FHFH
FFFH
HFFG


【执行动作 1 (向下) 后的地图】
  (Down)
SFFF
FHFH
FFFH
HFFG

  新状态索引: 0
  获得奖励: 0
  是否自然终止 (terminated): False
  是否超时截断 (truncated): False
  额外信息 (info): {'prob': 0.33333333333333337}


### 1. 地图上的字母代表什么？
*   **S (Start)**：起点，智能体的初始位置（左上角），安全。
*   **F (Frozen)**：结冰的安全湖面，可以正常行走。
*   **H (Hole)**：冰面上的洞，踩上去会掉进湖里，回合立刻结束（奖励为 0）。
*   **G (Goal)**：终点（右下角），到达即算成功，获得 +1 奖励。

### 2. 为什么执行动作后，地图看起来“没变化”？
这是新版 `gymnasium` 库中 `render_mode="ansi"` 的显示特性。它每次只打印**静态的地图地形**，并不会像游戏那样用一个特殊符号（比如红块）标出智能体当前具体站在哪个格子上。

所以，虽然地图字符看起来一模一样，但**智能体的真实位置其实已经改变了**。要确切知道它走到哪了，需要看代码里打印的**状态索引数字**：
*   地图按从左到右、从上到下编号为 **0 到 15**。
*   起点 **S** 是状态 **0**。
*   执行动作 1（向下）后，智能体移动到了正下方的格子，也就是第二行第一列，此时打印出的 `新状态索引` 应该是 **4**。

### 3. 为什么向下走没有掉进洞里？
观察地图第二行第一列，它是 **F**（安全冰面），而不是 **H**（洞）。所以智能体向下走了一步是安全的。

**⚠️ 核心难点提示：**
由于创建环境时设置了 `is_slippery=True`（冰面湿滑），智能体执行“向下”动作时，只有 1/3 的概率真的向下走，还有 1/3 的概率会向左滑，1/3 的概率向右滑。
在这个初始位置（状态 0），向左和向上都会撞墙（退回原地状态 0），只有向右会滑到状态 1。因此，即使你下达了“向下”的指令，它也有概率停在原地或滑向右边，这就是强化学习中**环境随机性**的直观体现。

强化学习中一个非常核心的概念—— **环境的随机性（Environment Stochasticity）**。

下面逐一拆解这些输出数据背后的真实含义：

### 1. 为什么“新状态索引”还是 0？
这正是因为我们在创建环境时保留了默认的 `is_slippery=True`（冰面湿滑）。
在 FrozenLake 环境中，当你下达一个动作指令时，智能体并不会乖乖听话，而是有 **1/3 的概率**执行你想要的方向，同时有 **1/3 的概率向左偏 90 度**，以及 **1/3 的概率向右偏 90 度**。

让我们回到起点（状态 0，左上角），当你下达动作 `1`（向下）时，实际发生的概率分布是这样的：
*   **向下走（预期动作）**：有 1/3 的概率成功走到状态 4。
*   **向左偏（逆时针旋转 90 度）**：有 1/3 的概率试图向左走。但因为智能体已经在最左侧边缘，**撞墙了**，所以只能退回原地，也就是状态 0。
*   **向右偏（顺时针旋转 90 度）**：有 1/3 的概率试图向右走，走到状态 1。

你这次运行代码，刚好触发了**“向左偏撞墙退回原地”**的分支，所以新状态依然是 0。

### 2. 为什么 `info` 里的 `prob` 是 0.33333333333333337？
这个字典是环境返回的额外诊断信息。这里的 `prob` 精确地告诉你：**本次状态转移所发生的实际概率**。
正如上面分析的，无论是向下走到 4，向左撞墙退回 0，还是向右滑到 1，它们发生的理论概率都是 **1/3**（即 0.3333...）。这个数字完美印证了环境底层的转移概率矩阵 $P$ 的设定。

### 3. 为什么其他输出也是正常的？
*   **获得奖励: 0**：FrozenLake 是一个典型的**稀疏奖励（Sparse Reward）**环境。只有在踩到终点 `G` 时才会给 +1 分，在冰面上正常行走（哪怕撞墙）都是 0 分。
*   **是否自然终止 (terminated): False**：智能体既没有掉进冰洞 `H`，也没有到达终点 `G`，游戏当然没有结束。
*   **是否超时截断 (truncated): False**：环境默认限制每个回合最多走 100 步，你才走了 1 步，远没有超时。

### 总结
你这次单步试探，就像是智能体在湿滑的冰面上尝试往下迈一步，结果脚底一滑，往左边打了个趔趄，但因为靠着墙，又跌回了原点。

**这正是强化学习需要解决的核心痛点**：在一个充满不确定性的物理世界里，智能体不能仅仅依赖“我下达指令 -> 环境必定执行”的确定性逻辑，它必须通过大量的试错（跑成千上万个回合），去计算每个动作的**期望收益**，从而学会在“滑溜溜”的环境中预判风险，找到真正能通向终点的最优策略。

In [6]:
# ==========================================
# 2. 核心函数：执行一个完整回合
# ==========================================

def run_episode(env, policy):
    """
    让智能体按照给定策略在环境中跑完一个完整回合
    """
    # 新版 API：reset 返回 (状态, 信息)
    state, _ = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        # 根据当前状态，从策略表中查出应该执行的动作
        action = policy[state].item()
        
        # 新版 API：step 返回 5 个值
        state, reward, terminated, truncated, info = env.step(action)
        
        # 累加奖励
        total_reward += reward
        
        # 新版终止逻辑：自然终止 或 超时截断，都代表回合结束
        done = terminated or truncated
        
    return total_reward

In [7]:
# ==========================================
# 3. 评估纯随机策略
# ==========================================

n_episode = 1000
total_rewards = []

print("\n" + "=" * 50)
print("【开始评估纯随机策略】")
print("=" * 50)

for episode in range(n_episode):
    # 生成一个完全随机的策略（每个状态随机分配一个动作）
    random_policy = torch.randint(high=n_action, size=(n_state,))
    total_reward = run_episode(env, random_policy)
    total_rewards.append(total_reward)

avg_reward_random = sum(total_rewards) / n_episode
print(f"在纯随机策略下，{n_episode} 个回合的平均总奖励为: {avg_reward_random:.4f}")


【开始评估纯随机策略】
在纯随机策略下，1000 个回合的平均总奖励为: 0.0180


In [8]:
# ==========================================
# 4. 随机搜索寻找“幸运策略”
# ==========================================

print("\n" + "=" * 50)
print("【开始随机搜索寻找能通关的幸运策略】")
print("=" * 50)

best_policy = None
# 不断生成随机策略，直到碰巧找到一个能拿到奖励 1（成功到达终点）的策略
while True:
    random_policy = torch.randint(high=n_action, size=(n_state,))
    total_reward = run_episode(env, random_policy)
    if total_reward == 1:
        best_policy = random_policy
        print("  [成功] 找到一个能通关的幸运策略！")
        break

# 用这个幸运策略再跑 1000 个回合，验证其稳定性
total_rewards = []
for episode in range(n_episode):
    total_reward = run_episode(env, best_policy)
    total_rewards.append(total_reward)

print(f"  幸运策略的具体动作分配: {best_policy}")
avg_reward_best = sum(total_rewards) / n_episode
print(f"在幸运策略下，{n_episode} 个回合的平均总奖励为: {avg_reward_best:.4f}")
print("  [说明] 由于冰面是滑的，即使是碰巧通关的策略，平均分也达不到 1.0，说明它并非真正的最优策略。")


【开始随机搜索寻找能通关的幸运策略】
  [成功] 找到一个能通关的幸运策略！
  幸运策略的具体动作分配: tensor([2, 1, 0, 0, 0, 3, 0, 0, 0, 0, 2, 1, 3, 2, 1, 2])
在幸运策略下，1000 个回合的平均总奖励为: 0.0330
  [说明] 由于冰面是滑的，即使是碰巧通关的策略，平均分也达不到 1.0，说明它并非真正的最优策略。


In [9]:
# ==========================================
# 5. 查看环境底层模型（转移概率矩阵 P）
# ==========================================

print("\n" + "=" * 50)
print("【查看环境底层转移概率矩阵 P】")
print("=" * 50)

# 使用 env.unwrapped.P 穿透包装器获取底层真实环境
print(f"状态 6 的转移概率分布:\n{env.unwrapped.P[6]}")
print(f"状态 11 的转移概率分布:\n{env.unwrapped.P[11]}")
print("  [说明] 每个动作都有 33.3% 的概率向三个不同方向滑动，这就是环境'滑溜溜'的数学本质。")

# 关闭环境释放资源
env.close()


【查看环境底层转移概率矩阵 P】
状态 6 的转移概率分布:
{0: [(0.33333333333333337, 2, 0, False), (0.3333333333333333, 5, 0, True), (0.33333333333333337, 10, 0, False)], 1: [(0.33333333333333337, 5, 0, True), (0.3333333333333333, 10, 0, False), (0.33333333333333337, 7, 0, True)], 2: [(0.33333333333333337, 10, 0, False), (0.3333333333333333, 7, 0, True), (0.33333333333333337, 2, 0, False)], 3: [(0.33333333333333337, 7, 0, True), (0.3333333333333333, 2, 0, False), (0.33333333333333337, 5, 0, True)]}
状态 11 的转移概率分布:
{0: [(1.0, 11, 0, True)], 1: [(1.0, 11, 0, True)], 2: [(1.0, 11, 0, True)], 3: [(1.0, 11, 0, True)]}
  [说明] 每个动作都有 33.3% 的概率向三个不同方向滑动，这就是环境'滑溜溜'的数学本质。


### 1. 为什么是 3 个方向，而不是 2 个？
你提到“6 不是有 2 个 H，2 个 F 吗”，这是从地图的**静态布局**来看的。但 FrozenLake 的滑动机制是基于**动作方向**的，而不是基于周围有几个洞。

在 `is_slippery=True` 的设定下，当你下达一个动作指令时，智能体只会向**三个方向**滑动：
1.  **1/3 的概率**：执行你期望的动作方向。
2.  **1/3 的概率**：向左偏 90 度（逆时针）。
3.  **1/3 的概率**：向右偏 90 度（顺时针）。

它**绝对不会**向后倒退（比如你按“上”，它不会滑向“下”）。所以，无论周围有几个洞，任何动作都只对应 3 个可能的转移方向。

---

### 2. 深度拆解：状态 6 的真实转移矩阵
让我们把状态 6（第二行第三列，安全冰面 F）的字典彻底翻译成人类语言，看看这 3 个方向到底去了哪里：

#### 动作 0：向左 (Left)
*   **1/3 概率** 真的向左走 ➡️ 到达状态 **2** (第一行第三列，F)，安全。
*   **1/3 概率** 逆时针偏转（变成向上走） ➡️ 到达状态 **5** (第二行第二列，**H**)，掉进冰洞！
*   **1/3 概率** 顺时针偏转（变成向下走） ➡️ 到达状态 **10** (第三行第三列，F)，安全。

#### 动作 1：向下 (Down)
*   **1/3 概率** 真的向下走 ➡️ 到达状态 **10** (第三行第三列，F)，安全。
*   **1/3 概率** 逆时针偏转（变成向右走） ➡️ 到达状态 **7** (第二行第四列，**H**)，掉进冰洞！
*   **1/3 概率** 顺时针偏转（变成向左走） ➡️ 到达状态 **5** (第二行第二列，**H**)，掉进冰洞！

#### 动作 2：向右 (Right)
*   **1/3 概率** 真的向右走 ➡️ 到达状态 **7** (第二行第四列，**H**)，掉进冰洞！
*   **1/3 概率** 逆时针偏转（变成向上走） ➡️ 到达状态 **2** (第一行第三列，F)，安全。
*   **1/3 概率** 顺时针偏转（变成向下走） ➡️ 到达状态 **10** (第三行第三列，F)，安全。

#### 动作 3：向上 (Up)
*   **1/3 概率** 真的向上走 ➡️ 到达状态 **2** (第一行第三列，F)，安全。
*   **1/3 概率** 逆时针偏转（变成向左走） ➡️ 到达状态 **5** (第二行第二列，**H**)，掉进冰洞！
*   **1/3 概率** 顺时针偏转（变成向右走） ➡️ 到达状态 **7** (第二行第四列，**H**)，掉进冰洞！

---

### 3. 为什么状态 11 只有 1 个方向？
你看到的第二个输出 `状态 11` 是地图第三行第四列，它是一个 **H（冰洞）**。
在 FrozenLake 中，一旦掉进冰洞，游戏就结束了。所以无论你按什么方向，它都会以 **100% 的概率**停留在状态 11，并且 `terminated=True`（回合终止）。这就是为什么它的字典里只有一个 `(1.0, 11, 0, True)`。

### 总结
状态 6 周围确实有 2 个洞（状态 5 和 7），但滑动机制是**基于动作方向的 3 个分支**，而不是“随机滑向周围的任意一个格子”。

从上面的拆解可以看出，状态 6 其实是个**非常危险**的位置：
*   按“向下”走，有 2/3 的概率掉进洞里（滑向 7 或 5）。
*   按“向上”走，也有 2/3 的概率掉进洞里（滑向 5 或 7）。
*   只有按“向左”或“向右”走，掉洞的概率才降到 1/3。

这就是强化学习的魅力所在：智能体必须通过计算这些转移概率，才能学会在危险区域选择最安全的动作！

这段代码的核心逻辑非常清晰，它是强化学习中 **基于随机策略的蒙特卡洛评估（Random Policy Evaluation）** 与 **随机搜索策略（Random Search Policy）** 的入门实现。

简单来说，它的任务是：**先让智能体在冰湖里瞎逛，看看随机乱走能拿多少分；然后暴力穷举出一把能通关的“运气流”策略，再验证它的稳定性。**

下面按照代码从上到下的顺序，拆解它**讲了什么（目的）**以及**怎么做的（实现原理）**：

---

### 第一部分：环境初始化与基础交互（第 1-25 行）
**讲了什么：** 搭建 4x4 的虚拟“冰湖”世界，并演示智能体如何与环境进行最基本的“一问一答”交互。
**怎么做的：**
1. **升级环境库**：将老旧的 `gym` 替换为官方继任者 `gymnasium`，并将环境升级为 `FrozenLake-v1`。
2. **获取空间维度**：通过 `observation_space.n` 和 `action_space.n` 获取状态总数（16个格子）和动作总数（4个方向）。
3. **重置与渲染**：调用 `env.reset()` 将智能体放回起点。注意新版 API 中 `reset()` 会返回 `(状态, 信息字典)` 两个值。
4. **单步试探**：调用 `env.step(1)` 让智能体执行“向下”的动作。新版 API 返回 5 个值：`(新状态, 奖励, 是否自然终止, 是否超时截断, 额外信息)`。

---

### 第二部分：核心函数 `run_episode`（第 27-37 行）
**讲了什么：** 定义一个完整的“回合（Episode）”执行流程，让智能体按照给定的策略从起点走到终点，并计算总得分。
**怎么做的：**
1. **初始化**：重置环境获取初始状态，初始化总奖励 `total_reward = 0`。
2. **循环交互**：在 `while not done` 循环中：
   * 根据传入的 `policy` 矩阵，查表获取当前状态应该执行的动作。
   * 调用 `env.step(action)` 推进环境。
   * 累加即时奖励。
   * **新版适配**：将 `terminated`（自然终止，如掉进冰洞或到达终点）和 `truncated`（超时截断）合并判断，只要有一个为 `True` 就结束本回合。

---

### 第三部分：评估纯随机策略（第 39-50 行）
**讲了什么：** 测试一个完全没有大脑、每次遇到新状态都随机瞎选方向的智能体，能拿到多少平均分。
**怎么做的：**
1. **生成随机策略**：使用 `torch.randint` 生成一个长度为 16 的张量，代表每个格子随机分配一个动作。
2. **批量测试**：循环 1000 个回合，每次都用这个随机策略跑完 `run_episode`。
3. **计算期望**：将 1000 次的总奖励求平均，得出随机策略下的期望收益（通常非常低，因为很容易掉进冰洞）。

---

### 第四部分：随机搜索寻找“幸运策略”（第 52-65 行）
**讲了什么：** 既然瞎逛不行，那就暴力穷举。不断生成新的随机策略，直到碰巧抽到一个能通关（总奖励为 1）的策略。
**怎么做的：**
1. **无限循环抽奖**：不断生成新的随机策略并执行一个回合。
2. **筛选幸运儿**：如果某个策略跑出来的 `total_reward == 1`（说明成功到达终点），就把它保存为 `best_policy` 并跳出循环。
3. **验证稳定性**：拿着这个“幸运策略”再跑 1000 个回合，看看它的平均分。
   * *预期结果*：由于冰湖环境是**随机滑动**的（`is_slippery=True`），即使是这个碰巧通关的策略，平均分也大概率达不到 1.0，证明它并不是真正的最优策略。

---

### 第五部分：查看环境底层模型（第 67-69 行）
**讲了什么：** 揭开环境的“底牌”，查看马尔可夫决策过程（MDP）中已知的状态转移概率矩阵 $P$。
**怎么做的：**
1. **穿透包装器**：使用 `env.unwrapped.P` 获取底层真实的转移矩阵。
2. **打印特定状态**：打印状态 6 和状态 11 的转移概率。你可以清晰地看到，对于每个动作，都有 33.3% 的概率向三个不同方向滑动，这正是环境“滑溜溜”的数学本质。

---

### 总结
这段代码**从上到下**完成了一个完整的“试错与碰运气”实验闭环：
**环境初始化 → 定义单回合执行器 → 评估纯随机策略 → 暴力搜索幸运策略 → 揭示环境底层转移概率**。

它没有涉及复杂的动态规划计算，而是通过**蒙特卡洛模拟**（跑很多回合求平均）来直观感受强化学习中“策略”与“环境随机性”的博弈。